In [ ]:
"""
Document Indexing Pipeline for Hybrid Search

This notebook builds the retrieval indexes used by the HybridSearchAgent:
  - Chroma vector store  : stores OpenAI-embedded document chunks for semantic search
  - SQLite FTS5 index    : stores raw chunk text for keyword (full-text) search

Workflow
--------
1. Load documents (.txt, .pdf, .docx) from the `documents/` directory.
2. Build a Pandas DataFrame to inspect and optionally enrich document-level metadata
   (filename, folder, file type, word/char counts, custom categories, timestamps).
3. Persist the DataFrame as CSV for reproducibility.
4. Convert back to LangChain Documents and split into overlapping chunks.
5. Annotate chunks with positional metadata (chunk_id, start_char, end_char).
6. Validate and standardize chunk metadata with a Pydantic model.
7. Embed chunks and upsert into Chroma.
8. Populate the SQLite FTS5 table used by the full-text search layer.
9. PCA visualization of the resulting embedding space (optional, 2-D scatter plot).

Run the cells top-to-bottom on first use; re-run individual sections to
refresh either index after document changes.
"""


In [ ]:
# Standard library
import os
from datetime import datetime

# Third-party
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader, Docx2txtLoader

# Load env from specific path
load_dotenv("C:/Users/kjosi/dotenv/.env")

### LOAD DOCUMENTS AND CREATE PANDAS DATAFRAME

In [ ]:
# Load documents with DirectoryLoader - loads multiple files from a specified directory into LangChain Document objects
txt_loader = DirectoryLoader("./documents", glob="**/*.txt", loader_cls=TextLoader)
pdf_loader = DirectoryLoader("./documents", glob="**/*.pdf", loader_cls=PyPDFLoader)
docx_loader = DirectoryLoader("./documents", glob="**/*.docx", loader_cls=Docx2txtLoader)

documents = txt_loader.load() + pdf_loader.load() + docx_loader.load()
print(f"Loaded {len(documents)} documents")

In [ ]:
# Print one of the documents
print(documents[0])

In [ ]:
# Create pandas dataframe from documents and metadata
rows = []
for doc in documents:
    rows.append({
        "text": doc.page_content,
        **doc.metadata
    })

df = pd.DataFrame(rows)
df.head()

### OPTIONS FOR ADDING METADATA

In [ ]:
## OPTIONAL STEP
# Drop columns
df = df.drop(["list_columns_to_dorp"], axis=1)
df.head()

In [ ]:
## OPTIONAL STEP
# Columns to keep - remove unwanted metadata columns
df = df[["text", "source"]]
df.head()

In [ ]:
## OPTIONAL STEP
# Extract filename and folder from source path
df["filename"] = df["source"].apply(lambda x: os.path.basename(x))
df["folder"] = df["source"].apply(lambda x: os.path.dirname(x))
df.head()

In [ ]:
## OPTIONAL STEP
# Add file type
df["file_type"] = df["filename"].apply(lambda x: os.path.split(".")[-1])
df.head()

In [ ]:
## OPTIONAL STEP
# Add text length
df["char_count"] = df["text"].apply(len)
df["word_count"] = df["text"].apply(lambda x: len(x.split()))
df.head()

In [ ]:
## OPTIONAL STEP
# Add custom tags - example: category
def classify_doc(text):
    if "finance" in text.lower():
        return "finance"
    return "general"

df["category"] = df["text"].apply(classify_doc)
df.head()

In [ ]:
## OPTIONAL STEP - BUG !
# Add ingestion timestamp
df["ingested_at"] = datetime.utcnow()
df.head()

##!! Must make it str or int or something that is accepted during embedding

In [ ]:
## OPTIONAL STEP
# Save df for temporary storage as csv, save for reproducibility
df.to_csv("documents_metadata_df.csv", index=False)

# Load file later
df = pd.read_csv("documents_metadata_df.csv")

In [ ]:
## OPTIONAL STEP - BUG !
# Save df for temporary storage as parquet file, save for reproducibility
df.to_parquet("documents_with_metadata.parquet", index=False)

# Load file later
df = pd.read_parquet("document_with_metadata.parquet")

### CONVERT BACK FROM PANDAS DATAFRAME TO DOCUMENTS

In [ ]:
# Convert back to LangChain Document objects with metadata
documents_metadata = []

for _, row in df.iterrows(): 
    metadata = row.drop("text").to_dict()

    documents_metadata.append(
        Document(
            page_content=row["text"],
            metadata=metadata
        )
    )

print(f"Loaded {len(documents)} documents")

In [ ]:
# Print one of the documents
print(documents_metadata[0])

In [ ]:
## OPTIONAL STEP
# If small tweaks are needed, change metadata after document creation, example: category
for doc in documents_metadata:
    doc.metadata["category"] = "new_value"

### CREATE CHUNKS AND ADD CHUNK METADATA

In [ ]:
# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents_metadata)

In [ ]:
# Print chunk metadata
for i, chunk in enumerate(chunks):
    print(chunk.metadata)

In [ ]:
# Print specific chunk metadata, example: source
for i, chunk in enumerate(chunks):
    print(chunk.metadata["source"])

In [ ]:
# Add chunk metadata, example: chunk_id
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i


In [ ]:
# Add chunk metadata, example: real position tracking
start = 0

for i, chunk in enumerate(chunks):
    text = chunk.page_content
    end = start + len(text)

    chunk.metadata["start_char"] = start
    chunk.metadata["end_char"] = end

    start = end

In [ ]:
# Print chunk metadata
for i, chunk in enumerate(chunks):
    print(chunk.metadata)

In [ ]:
# Print specific chunk metadata, example: source
for i, chunk in enumerate(chunks):
    print(chunk.metadata["source"])

### STANDARDIZE CHUNK METADATA

In [ ]:
# Definition to standardize chunk metadata
from pydantic_models import ChunkMetadata

def standardize_chunk_metadata(chunk):
    """
    Standardizes chunk metadata using Pydantic for type validation.

    Parses metadata into structured fields (e.g., "3" -> 3) based on 
    ChunkMetadata and isolates any remaining extra fields.

    Returns:
        tuple: (original chunk, structured_dict, extra_fields_dict)
    """
    
    # Explain, example: "chunk_id": "3" > Pydantic converts it > 3 (int)
    std_meta = ChunkMetadata.model_validate(chunk.metadata)

    # model_dump() converts pydandic model into a standard python dict
    structured_metadata = std_meta.model_dump() 

    # Extracts "extra metadata", everything that is NOT in the schema
    extra_metadata = {key: value for key, value in chunk.metadata.items() if key not in structured_metadata}

    return chunk, structured_metadata, extra_metadata

In [ ]:
# Run standardize_chunk_metadata() function on all chunks
clean_chunks = []

for chunk in chunks:
    _, structured_metadata, extra_metada = standardize_chunk_metadata(chunk)

    clean_chunk = chunk.model_copy()
    clean_chunk.metadata = {**structured_metadata, **extra_metada}

    clean_chunks.append(clean_chunk)

### INGEST CHUNKS: CHROMA DB AND SQLite FTS

In [ ]:
# Create / load Chroma DB (persisted locally) - when DB does not already exist
vector_store = Chroma(
    collection_name="document_collection_1",
    embedding_function=OpenAIEmbeddings(),
    persist_directory="./chroma_db"
)

# Add chunks to Chroma db
vector_store.add_documents(clean_chunks)

print("Embeddings stored in ./chroma_db")

In [ ]:
## OPTIONAL STEP — UPDATE CHROMA DB (delete + upsert)
# Use this cell instead of the one above when the DB already exists.
# It can do a source-level refresh before upsert:
#   1) delete existing chunks for the same source
#   2) upsert current chunks with deterministic IDs
# This avoids stale chunks if a document now produces fewer chunks than before.

import hashlib

# Set to True when you want to fully refresh all sources present in clean_chunks.
# Set to False to skip delete and do pure upsert.
REFRESH_EXISTING_SOURCES = True

# Load existing Chroma DB
vector_store = Chroma(
    collection_name="document_collection_1",
    embedding_function=OpenAIEmbeddings(),
    persist_directory="./chroma_db"
)

# Build deterministic chunk IDs from source path + chunk_id
def make_chunk_id(chunk):
    key = f"{chunk.metadata.get('source', '')}_{chunk.metadata.get('chunk_id', '')}"
    return hashlib.md5(key.encode()).hexdigest()

texts = [chunk.page_content for chunk in clean_chunks]
ids = [make_chunk_id(chunk) for chunk in clean_chunks]
metadatas = [chunk.metadata for chunk in clean_chunks]

# Optional cleanup: remove old rows for the same sources before re-inserting/updating
if REFRESH_EXISTING_SOURCES:
    sources_to_refresh = sorted({m.get("source") for m in metadatas if m.get("source")})
    for src in sources_to_refresh:
        vector_store._collection.delete(where={"source": src})
    print(f"Deleted existing chunks for {len(sources_to_refresh)} source(s)")

# Compute embeddings and upsert into the underlying Chroma collection
embeddings = OpenAIEmbeddings().embed_documents(texts)

vector_store._collection.upsert(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadatas
)

print(f"Upserted {len(clean_chunks)} chunks into Chroma DB")


In [ ]:
# Initiate FTS and index chunks - when FTS index does not already exist
from fts_search import FTSStore

fts_store = FTSStore()
fts_store.add_documents(clean_chunks)

print("Indexing complete")

In [ ]:
## OPTIONAL STEP — UPDATE SQLite FTS (delete + re-index)
# Use this cell instead of the one above when the FTS index already exists.
# It can do a source-level refresh before insert:
#   1) delete existing chunks for the same source
#   2) insert current chunks for those/new sources
# This avoids stale rows if a document now produces fewer chunks than before.

# Set to True when you want to fully refresh all sources present in clean_chunks.
# Set to False to only append chunks from sources not already indexed.
REFRESH_EXISTING_SOURCES = True

# Load existing FTS store
fts_store = FTSStore()

# Sources present in the current batch
batch_sources = sorted({chunk.metadata.get("source") for chunk in clean_chunks if chunk.metadata.get("source")})

if REFRESH_EXISTING_SOURCES:
    # Delete existing rows for each source in the current batch
    for src in batch_sources:
        fts_store.cur.execute("DELETE FROM docs WHERE source = ?", (src,))
    fts_store.conn.commit()

    # Re-index the current batch
    fts_store.add_documents(clean_chunks)
    print(f"SQLite FTS refreshed for {len(batch_sources)} source(s); added {len(clean_chunks)} chunks")
else:
    # Append mode: only add chunks whose source is not yet indexed
    fts_store.cur.execute("SELECT DISTINCT source FROM docs")
    indexed_sources = {row[0] for row in fts_store.cur.fetchall()}

    new_chunks = [
        chunk for chunk in clean_chunks
        if chunk.metadata.get("source") not in indexed_sources
    ]

    if new_chunks:
        fts_store.add_documents(new_chunks)
        print(f"SQLite FTS updated — added {len(new_chunks)} new chunks")
    else:
        print("SQLite FTS is already up to date — no new chunks to add")


### CHECK DOCUMENTS, CHUNKS, AND METADATA

In [ ]:
# View what is stored in persisted vector store
vector_store = Chroma(
    collection_name="document_collection_1",
    embedding_function=OpenAIEmbeddings(),
    persist_directory="./chroma_db"
)

print(vector_store.get())

In [ ]:
# Print chunk metadata
for i, chunk in enumerate(clean_chunks):
    print(chunk.metadata)

In [ ]:
# Print and inspect metadata
collection = vector_store._collection
data = collection.get(include=["embeddings", "documents", "metadatas"])

print(f"Number of chunks: {len(data['documents'])}\n")

for i in range(len(data["documents"])):
    print("ID:", data["ids"][i])
    print("TEXT:", data["documents"][i])
    print("METADATA:", data["metadatas"][i])
    print("VECTOR (first 5 dims):", data["embeddings"][i][:5])
    print("-"*40)

In [ ]:
# Test retrieval, metatdata
results = vector_store.similarity_search("your query", k=1)

for result in results:
    print(result.page_content)
    print(result.metadata)

In [ ]:
# Run PCA to see how documents cluster - under development !!
data = collection.get(include=["embeddings"])
embeddings = np.array(data["embeddings"])

pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)